# 04 — ML Analysis: Classification & RL

1. Supervised classification: predict sex / breeding status / network role  
2. Feature importance from Random Forest  
3. RL environment demo: random policy on `BirdColonyEnv`  

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from ml.features import build_feature_matrix, GRAPH_FEATURE_COLS
from ml.classification import train_and_evaluate, evaluate_all_targets
from ml.rl.environment import BirdColonyEnv

YEAR   = 2016
PLOT   = 'SPRA'
PERIOD = 'daytime'

## 1. Build feature matrix

In [ ]:
# Combined graph metrics + embeddings
X, feat_names, meta = build_feature_matrix(
    YEAR, PLOT, PERIOD,
    use_graph_metrics=True,
    use_embeddings=True,
    normalise=True,
)
print(f'Feature matrix: {X.shape}  ({X.shape[1]} features)')
print(f'Available labels: {[c for c in meta.columns if c != "bird_id"]}')
meta.head()

## 2. Supervised classification

In [ ]:
# Add network_position as a target (from nodes file)
from config import OUTPUT_ROOT
label = f'{YEAR}_{PLOT}_{PERIOD}'
nodes = pd.read_csv(OUTPUT_ROOT / label / f'nodes_{label}.csv')
if 'network_position' in nodes.columns:
    meta = meta.merge(nodes[['bird_id', 'network_position']], on='bird_id', how='left')
meta.head()

In [ ]:
# Run all available targets
all_results = evaluate_all_targets(X, meta, feature_names=feat_names)
for target, res in all_results.items():
    print(f'\n=== {target} ===')
    print(res['summary'].to_string(index=False))

## 3. Confusion matrix (best model for sex)

In [ ]:
if 'sex' in all_results:
    res = all_results['sex']
    best_model_name = res['summary'].iloc[0]['model']
    cm = res[best_model_name]['confusion_matrix']
    labels = res['label_names']

    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=np.array(cm), display_labels=labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{best_model_name}\nROC-AUC = {res[best_model_name]["roc_auc"]}')
    plt.tight_layout()
    plt.show()
else:
    print('Sex labels not available.')

## 4. Feature importance (Random Forest)

In [ ]:
for target in ('sex', 'network_position', 'breeding_status'):
    if target not in all_results:
        continue
    fi = all_results[target].get('feature_importance')
    if fi is None:
        continue
    top = fi.head(20)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'Top-20 features — target: {target}')
    plt.tight_layout()
    plt.show()
    break   # show just the first available target

## 5. RL Environment Demo

The `BirdColonyEnv` models a bird choosing a nest/roost site at each timestep.  
Reward = SRI association strength with the bird at the chosen nest.  
Here we run a **random policy** to verify the env works end-to-end.

In [ ]:
env = BirdColonyEnv.from_context(YEAR, PLOT, PERIOD, seed=42)
print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)
print('Number of birds:  ', env._n_birds)

In [ ]:
# Run one episode with a random policy
obs, info = env.reset()
rewards = []

done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    done = terminated or truncated

print(f'Episode length: {len(rewards)} steps')
print(f'Mean reward:    {np.mean(rewards):.4f}')
print(f'Max reward:     {np.max(rewards):.4f}')
print(f'Non-zero steps: {np.sum(np.array(rewards) > 0)}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(rewards, alpha=0.7, color='mediumseagreen')
ax.axhline(np.mean(rewards), color='red', ls='--', label=f'mean={np.mean(rewards):.3f}')
ax.set_xlabel('Step')
ax.set_ylabel('Reward (SRI)')
ax.set_title('Random policy episode rewards')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# The env is Gymnasium-compatible — plug in any agent from stable-baselines3, etc.
# Example (requires: pip install stable-baselines3):
#
# from stable_baselines3 import PPO
# model = PPO('MlpPolicy', env, verbose=1)
# model.learn(total_timesteps=10_000)
print('Ready for agent training — see comment above for stable-baselines3 example.')